# 1. Vom Tabellen-Netz zum Bild-Netz — Computer Vision beim Kfz-Versicherer

**Session 7 · Dauer: 30–60 Min**

## Real-World-Kontext

In Kapitel 6 hast Du ein neuronales Netz (`VersichererMLP`) für **Tabellendaten** gebaut —
Fahreralter, Schadenshistorie, Fahrzeugtyp. Am Ende dieses Kapitels stand ein ehrliches Fazit:
Für 350 Fälle mit drei Spalten lohnt sich Deep Learning eigentlich nicht — Random Forest aus
Kapitel 4 ist da die bessere Wahl.

Heute wechselt der Fall: **Bilder**. Und bei Bildern kippt die Rechnung komplett — genau die Sorte
Daten, für die Deep Learning erfunden wurde. Der Kfz-Versicherer aus unserem Case bekommt nicht
nur Tabellen, sondern auch **Schadensfotos** von Kunden eingereicht: Kratzer, Beulen,
Totalschäden. Ein Foto lässt sich nicht sinnvoll in eine Handvoll Tabellenspalten pressen — man
braucht ein Netz, das die **räumliche Struktur** von Pixeln versteht.

Wir bauen dieses Kapitel in zwei Teilen auf:

1. **MNIST als Lehrbuch-Beispiel** — der Klassiker der Bilderkennung (handschriftliche Ziffern
   0–9). An MNIST lernst Du die komplette CNN-Mechanik: Convolution, Pooling, Architektur,
   Training. Klein genug, um in wenigen Minuten auf einem Laptop zu trainieren.
2. **Kfz-Schadensfotos als konzeptuelles Transfer-Learning-Beispiel** — zurück zu unserem
   Running Example. Da uns kein echter Foto-Datensatz vorliegt, demonstrieren wir die
   **Transfer-Learning-API** (ein vortrainiertes ResNet50 laden, einfrieren, Custom Head
   ersetzen) an synthetischen Bild-Tensoren — sauber gekennzeichnet als didaktisches Mock, aber
   mit echtem, lauffähigem PyTorch-Code.

> 💡 **Good to know:**
> Warum zwei Beispiele parallel? MNIST zeigt Dir das **volle Training von Grund auf** (das
> brauchst Du, um CNNs wirklich zu verstehen). Kfz-Schadensfotos zeigen Dir, wie man in der
> **Praxis** vorgeht, wenn man — wie fast immer — nicht genug eigene Trainingsdaten hat: Man
> nutzt ein bereits trainiertes Netz und passt nur die letzte Schicht an.

## 🎯 Learning Objectives

By completing this notebook, you will be able to:
- 💡 **Explain** was eine **Convolution** (Filter, Kernel, Feature Map) und ein **Pooling**
  (Max-Pooling) rechnerisch tun, und beide Operationen an einem kleinen Beispiel von Hand
  nachrechnen.
- 🔍 **Identify** die Architektur eines einfachen CNNs (`SimpleCNN`: Conv → Pool → Conv → Pool →
  Flatten → Dense → Dense) und wissen, wie sich die räumliche Dimension von Schicht zu Schicht
  verändert.
- 🛠️ **Apply** `PyTorch` (`nn.Conv2d`, `nn.MaxPool2d`), um ein CNN für ein echtes
  Bildklassifikationsproblem (MNIST-Ziffernerkennung) zu definieren und zu trainieren.
- 🛠️ **Apply** den vollständigen CNN-Trainingsloop über mehrere Epochen und die Auswertung mit
  Test-Accuracy und Confusion Matrix.
- 🔍 **Identify** das Grundprinzip von **Transfer Learning** (Conv-Layer einfrieren, Custom Head
  ersetzen) und den zugehörigen PyTorch-Code an einem ResNet50-Beispiel nachvollziehen.
- ⚖️ **Bewerten**, wann sich ein CNN von Grund auf lohnt und wann Transfer Learning in der Praxis
  die bessere Wahl ist.

## Concept at a Glance

**Analogie — der Filter als Lupen-Scanner:** Stell Dir einen **Filter** (auch Kernel genannt) als
eine kleine, quadratische Lupe vor (z. B. 3×3 Pixel groß), die systematisch über ein Foto gleitet.
An jeder Position schaut die Lupe sich nur den kleinen Ausschnitt darunter an, multipliziert jeden
Pixelwert mit einem eigenen Gewicht und summiert alles zu **einer einzigen Zahl** auf. Diese Zahl
sagt: "Wie stark ist das Muster, nach dem diese Lupe sucht (z. B. eine Kante), genau hier
vorhanden?" Wandert die Lupe über das ganze Bild, entsteht eine neue, kleinere Karte aus lauter
solchen Zahlen — die **Feature Map**.

**Warum nicht einfach ein Dense-Netz wie in Kapitel 6?** Ein 224×224-RGB-Foto hat 150.528
Eingabewerte. Eine einzige Dense-Schicht mit 128 Neuronen bräuchte dafür **19,3 Millionen**
Gewichte. Ein Conv-Layer mit 32 Filtern der Größe 3×3 braucht dagegen nur **288** Gewichte —
weil derselbe kleine Filter wiederverwendet wird, statt für jedes einzelne Pixel ein eigenes
Gewicht zu lernen. Das ist eine Reduktion um rund das **67.000-fache**.

$$\text{Output}[i,j] = \sum_{a=0}^{k-1} \sum_{b=0}^{k-1} \text{Filter}[a,b] \times \text{Input}[i+a, j+b] + \text{Bias}$$

Nach der Convolution kommt meist **Pooling** — eine Verdichtung, die die räumliche Auflösung
reduziert (z. B. Max-Pooling: der größte Wert pro 2×2-Fenster gewinnt) und dabei kleine
Verschiebungen im Bild toleranter macht.

> 💡 **Good to know:**
> Bei einem Hand-designten Filter (wie dem Sobel-Filter, den wir gleich nachrechnen) legen *wir*
> die Gewichte fest. In einem echten CNN sind die Filter-Gewichte dagegen **trainierbar** — das
> Netz lernt selbst per Backpropagation, welche Filter für die jeweilige Aufgabe nützlich sind.

## Schritt 1 — Convolution von Hand nachrechnen

Wir rechnen jetzt **exakt das Beispiel von der Vorlesungsfolie** "Sobel-Filter anwenden: Das
Rechenbeispiel" nach: ein 3×3-Bildausschnitt (Patch) wird mit einem **Sobel-Filter** (erkennt
vertikale Kanten) element-weise multipliziert und aufsummiert.

$$\text{Patch} = \begin{bmatrix} 1 & 2 & 1 \\ 0 & 1 & 2 \\ 1 & 0 & 1 \end{bmatrix}
\qquad
\text{Sobel} = \begin{bmatrix} -1 & 0 & 1 \\ -2 & 0 & 2 \\ -1 & 0 & 1 \end{bmatrix}$$

In [ ]:
# I DO: Convolution eines 3x3-Patches mit dem Sobel-Filter (vertikale Kanten)

import numpy as np

# Bildausschnitt (Patch) aus der Vorlesungsfolie
patch = np.array([
    [1, 2, 1],
    [0, 1, 2],
    [1, 0, 1],
])

# Sobel-Filter (vertikal) - erkennt Spalten-Differenzen (linke vs. rechte Seite)
sobel_vertikal = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1],
])

# Element-weise Multiplikation (nicht Matrixmultiplikation!) und anschliessende Summe
produkt = patch * sobel_vertikal
output = produkt.sum()

print("Element-weises Produkt:\n", produkt)
print(f"\nOutput (Summe aller Produkte) = {output}")

### Schritt-für-Schritt-Breakdown

1. `patch * sobel_vertikal` (mit `numpy`-Arrays) multipliziert **jede Zelle einzeln** mit der
   Zelle an derselben Position — das ist **nicht** die Matrixmultiplikation, die Du aus der
   linearen Algebra kennst!
2. `.sum()` addiert alle neun Produkte zu **einer einzigen Zahl** — genau der Formel
   $\sum_{a} \sum_{b} \text{Filter}[a,b] \times \text{Input}[i+a,j+b]$ folgend (Bias hier bewusst
   weggelassen, wie auf der Vorlesungsfolie).
3. Das Ergebnis **4** deckt sich mit der Vorlesungsfolie: ein positiver, betragsmäßig recht hoher
   Wert bedeutet "hier ist eine starke vertikale Kante".

> 💡 **Good to know:**
> In einem echten Bild würde dieser Filter jetzt eine Position **weiterrutschen** (gesteuert
> durch den **Stride**) und die Rechnung an der nächsten Position wiederholen — so entsteht
> Stück für Stück die komplette **Feature Map**.

Jetzt probierst Du selbst aus, was mit einem anderen Patch passiert.

> 🎯 **Your Task:**
> Verändere in der Zelle unten den `patch_neu` (z. B. eine Spalte komplett auf denselben Wert
> setzen, sodass **keine** Kante mehr vorliegt) und beobachte, wie sich der Output verändert.

In [ ]:
# WE DO: Anderen Patch einsetzen und Effekt auf den Convolution-Output beobachten

patch_neu = np.array([
    [5, 5, 5],   # <- veraendere hier die Werte, z.B. alle Spalten gleich setzen (keine Kante)
    [5, 5, 5],
    [5, 5, 5],
])

output_neu = (patch_neu * sobel_vertikal).sum()
print(f"Neuer Output = {output_neu} (vorher: {output})")

> ⚠️ **Common Pitfall:**
> Ein `patch_neu` mit **überall demselben Wert** (kein Helligkeitsunterschied zwischen linker und
> rechter Seite) ergibt beim Sobel-Filter immer **0** — logisch, denn der Filter sucht ja explizit
> nach *Unterschieden* zwischen links und rechts. Keine Kante, kein Signal.

### Zwischenfazit

Du hast die Convolution-Operation jetzt von Hand nachgerechnet — genau die Rechnung, die ein CNN
tausendfach parallel für jede Filter-Position im gesamten Bild durchführt. Als Nächstes schauen
wir uns Pooling an, den zweiten zentralen Baustein.

## Schritt 2 — Pooling von Hand nachrechnen

Nach einer Convolution-Schicht folgt in einem CNN meist **Pooling** — eine Downsampling-Operation,
die die räumliche Größe reduziert. Wir rechnen wieder das Beispiel von der Vorlesungsfolie
"Pooling: Das Rechenbeispiel" nach: eine 4×4-Matrix wird mit **Max-Pooling(2×2)** auf eine 2×2-Matrix
verdichtet.

$$\begin{bmatrix} 1 & 3 & 2 & 7 \\ 0 & 5 & 1 & 4 \\ 2 & 1 & 6 & 2 \\ 8 & 3 & 5 & 1 \end{bmatrix}
\xrightarrow{\text{Max-Pool(2×2)}}
\begin{bmatrix} ? & ? \\ ? & ? \end{bmatrix}$$

In [ ]:
# I DO: Max-Pooling(2x2) von Hand ueber eine 4x4-Matrix

feature_map = np.array([
    [1, 3, 2, 7],
    [0, 5, 1, 4],
    [2, 1, 6, 2],
    [8, 3, 5, 1],
])

# Wir teilen die 4x4-Matrix in vier 2x2-Fenster auf und nehmen jeweils den Max-Wert
fenster_oben_links = feature_map[0:2, 0:2]
fenster_oben_rechts = feature_map[0:2, 2:4]
fenster_unten_links = feature_map[2:4, 0:2]
fenster_unten_rechts = feature_map[2:4, 2:4]

gepoolt = np.array([
    [fenster_oben_links.max(), fenster_oben_rechts.max()],
    [fenster_unten_links.max(), fenster_unten_rechts.max()],
])

print("Fenster oben links:\n", fenster_oben_links, "-> Max:", fenster_oben_links.max())
print("Fenster oben rechts:\n", fenster_oben_rechts, "-> Max:", fenster_oben_rechts.max())
print("\nErgebnis nach Max-Pool(2x2):\n", gepoolt)

> 💡 **Good to know:**
> Das Ergebnis `[[5, 7], [8, 6]]` deckt sich exakt mit der Vorlesungsfolie. Aus 16 Werten wurden
> **4** — eine **4-fache räumliche Reduktion**, ganz ohne trainierbare Parameter (Pooling hat
> keine Gewichte!). In einem CNN kommt diese Verdichtung mehrfach hintereinander vor: Nach zwei
> Pool(2×2)-Schritten ist ein Bild schon auf 1/16 seiner ursprünglichen Fläche geschrumpft.

Jetzt probierst Du selbst eine andere Fenstergröße aus.

> 🎯 **Your Task:**
> Ersetze in der Zelle unten `numpy`s eingebaute Pooling-Logik (statt der Handrechnung oben) und
> beobachte den Unterschied, wenn Du **Average-Pooling** statt Max-Pooling verwendest — nimm dafür
> `.mean()` statt `.max()` für jedes Fenster.

In [ ]:
# WE DO: Average-Pooling statt Max-Pooling ausprobieren

gepoolt_average = np.array([
    [fenster_oben_links.mean(), fenster_oben_rechts.mean()],
    [fenster_unten_links.mean(), fenster_unten_rechts.mean()],
])

print("Max-Pooling-Ergebnis:\n", gepoolt)
print("\nAverage-Pooling-Ergebnis:\n", gepoolt_average)

> ⚠️ **Common Pitfall:**
> Max-Pooling behält immer den **stärksten** Aktivierungswert in einem Fenster — das ist meist
> das gewünschte Verhalten, weil starke Aktivierungen ("hier ist eindeutig eine Kante") wichtiger
> sind als schwache. Average-Pooling glättet dagegen alles gleichmäßig und "verwässert" starke
> Signale — deshalb ist Max-Pooling in der Praxis der deutlich häufigere Standard.

### Zwischenfazit

Du kennst jetzt beide Kernoperationen eines CNNs von Hand: **Convolution** (Muster erkennen) und
**Pooling** (verdichten). Zeit, sie in einer echten Architektur zusammenzusetzen und an einem
echten Bild-Datensatz zu trainieren.

## Schritt 3 — MNIST laden und explorieren

**MNIST** ist der Klassiker unter den Bild-Datensätzen: 70.000 handgeschriebene Ziffern (0–9),
je 28×28 Pixel in Graustufen — 60.000 zum Trainieren, 10.000 zum Testen. Seit 1998 der
Standard-Benchmark für allererste CNN-Experimente: klein genug für einen Laptop, aber komplex
genug, um die volle CNN-Mechanik zu zeigen.

`torchvision` bringt MNIST direkt mit — kein manueller Download nötig.

In [ ]:
# I DO: MNIST-Datensatz laden (wird beim ersten Aufruf automatisch heruntergeladen)

from torchvision import datasets, transforms

# ToTensor() wandelt Bilder (0-255) in Tensoren im Bereich [0, 1] um.
# Normalize() zentriert die Werte zusaetzlich um den MNIST-Mittelwert/-Standardabweichung
# (0.1307, 0.3081 sind feste, aus dem Trainingsset vorab berechnete Konstanten).
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_dataset = datasets.MNIST("./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST("./data", train=False, download=True, transform=transform)

print(f"Trainingsbilder: {len(train_dataset)}")
print(f"Testbilder:      {len(test_dataset)}")

bild, label = train_dataset[0]
print(f"Form eines einzelnen Bild-Tensors: {tuple(bild.shape)}  (Kanaele, Hoehe, Breite)")
print(f"Label des ersten Bildes: {label}")

> 💡 **Good to know:**
> `tuple(bild.shape)` gibt `(1, 28, 28)` aus — **1 Kanal** (Graustufen, kein RGB), **28×28**
> Pixel. PyTorch erwartet Bild-Tensoren im **Channels-First-Format** (Kanäle zuerst), anders als
> z. B. `matplotlib`, das Channels-Last erwartet — deshalb müssen wir beim Plotten gleich einmal
> kurz umformen.

Schauen wir uns ein paar Beispielziffern direkt an.

In [ ]:
# I DO: Ein paar Beispielziffern visualisieren

import matplotlib.pyplot as plt

fig, achsen = plt.subplots(2, 5, figsize=(10, 4))

for i, achse in enumerate(achsen.flat):
    bild, label = train_dataset[i]
    # squeeze() entfernt die Kanal-Dimension (1, 28, 28) -> (28, 28), die matplotlib nicht braucht
    achse.imshow(bild.squeeze(), cmap="gray")
    achse.set_title(f"Label: {label}")
    achse.axis("off")

plt.tight_layout()
plt.show()

### Zwischenfazit

Wir haben jetzt einen echten, geladenen Bild-Datensatz vor uns — 60.000 Trainingsbeispiele,
normiert und bereit für ein CNN. Jetzt bauen wir das Netz selbst.

## Schritt 4 — `SimpleCNN` in PyTorch definieren

Jetzt übersetzen wir die Architektur von der Vorlesungsfolie "SimpleCNN in PyTorch — Architektur"
1:1 in Code:

- **Conv1:** `Conv2d(1, 32, kernel_size=3, padding=1)` — 1 Eingabekanal (Graustufen) → 32 Filter
- **Pool:** `MaxPool2d(2, 2)` — halbiert Höhe und Breite
- **Conv2:** `Conv2d(32, 64, kernel_size=3, padding=1)` — 32 → 64 Filter
- **Pool:** noch einmal halbiert
- **Flatten + Dense(128):** die verbliebene 7×7×64-Feature-Map wird zu einem Vektor "plattgemacht"
  und in eine Dense-Schicht mit 128 Neuronen gespeist
- **Dense(10):** Output-Schicht — eine Roh-Ausgabe (Logit) pro Ziffer (0–9)

**Räumlicher Verlauf:** 28×28 → (Pool) 14×14 → (Pool) 7×7. Das ist auch, woher `7 * 7 * 64` in
`fc1` kommt — genau die Größe der letzten Feature Map, plattgemacht.

In [ ]:
# I DO: SimpleCNN als eigene Klasse definieren

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)  # fester Seed -> reproduzierbare Gewichts-Initialisierung

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)   # 1 -> 32 Filter, Bildgroesse bleibt (padding=1)
        self.pool = nn.MaxPool2d(2, 2)                             # halbiert Hoehe und Breite
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)  # 32 -> 64 Filter
        self.fc1 = nn.Linear(7 * 7 * 64, 128)                      # geflattete Feature Map -> 128 Neuronen
        self.fc2 = nn.Linear(128, 10)                              # 128 -> 10 Ziffern-Klassen

    def forward(self, x):
        x = F.relu(self.conv1(x))   # Conv1 + ReLU: 28x28x1 -> 28x28x32
        x = self.pool(x)            # Pool: 28x28x32 -> 14x14x32
        x = F.relu(self.conv2(x))   # Conv2 + ReLU: 14x14x32 -> 14x14x64
        x = self.pool(x)            # Pool: 14x14x64 -> 7x7x64
        x = x.view(x.size(0), -1)   # Flatten: (Batch, 7, 7, 64) -> (Batch, 7*7*64)
        x = F.relu(self.fc1(x))     # Dense(128) + ReLU
        return self.fc2(x)          # Dense(10) -> Roh-Ausgaben (Logits), noch kein Softmax

model = SimpleCNN()
print(model)

anzahl_parameter = sum(p.numel() for p in model.parameters())
print(f"\nAnzahl trainierbarer Parameter: {anzahl_parameter:,}")

> 💡 **Good to know:**
> `padding=1` bei einem 3×3-Filter sorgt dafür, dass die Bildgröße **nach der Convolution
> gleich bleibt** (28×28 bleibt 28×28) — erst das nachfolgende `MaxPool2d(2, 2)` halbiert die
> räumliche Größe. Ohne Padding würde jede Convolution das Bild etwas schrumpfen lassen.

> ⚠️ **Common Pitfall:**
> `x.view(x.size(0), -1)` ist der häufigste Stolperstein bei CNNs: Die `-1` sagt PyTorch "rechne
> diese Dimension selbst aus", `x.size(0)` erhält aber explizit die **Batch-Größe** — sonst würde
> ein ganzer Batch versehentlich zu einem einzigen, viel zu langen Vektor plattgemacht.

Mit `nn.CrossEntropyLoss()` (für **10** Klassen, nicht binär wie in Kapitel 6) und dem
Adam-Optimizer sind wir bereit für den Trainingsloop.

## Schritt 5 — Daten vorbereiten & Trainingsloop

Zunächst packen wir Trainings- und Testdaten in `DataLoader`, die uns automatisch Batches
liefern — genau wie in Kapitel 6.

In [ ]:
# I DO: DataLoader fuer Training und Test

from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

print(f"Anzahl Trainings-Batches: {len(train_loader)}")
print(f"Anzahl Test-Batches:      {len(test_loader)}")

Jetzt kommt das Muster aus der Vorlesungsfolie "MNIST trainieren — Die Trainingsloop": derselbe
Fünf-Schritte-Ablauf wie in Kapitel 6 (Forward Pass → Loss → `zero_grad()` → `backward()` →
`step()`), nur diesmal mit `CrossEntropyLoss` für **10** Klassen statt `BCEWithLogitsLoss` für
eine binäre Entscheidung.

> 🎯 **Your Task:**
> Vervollständige die `# TODO`-Zeilen im Trainingsloop unten. Orientiere Dich am Muster aus
> Kapitel 6 bzw. der Vorlesungsfolie "MNIST trainieren — Die Trainingsloop".

In [ ]:
# =========================================================================
# 🎯 EXERCISE: Trainingsloop vervollstaendigen
# =========================================================================
# Instruction: Ersetze die TODO-Kommentare durch gueltigen Code (siehe Anleitung oben).

torch.manual_seed(42)
model = SimpleCNN()
criterion = nn.CrossEntropyLoss()                     # Standard-Loss fuer Multiklassen-Klassifikation
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

anzahl_epochen = 5
loss_pro_epoche = []

for epoch in range(anzahl_epochen):
    model.train()  # Trainingsmodus (wichtig bei manchen Layern, hier v.a. gute Praxis)
    epoch_losses = []

    for batch_x, batch_y in train_loader:
        # 1. Forward Pass: Vorhersage des Modells fuer diesen Batch berechnen
        output = ???

        # 2. Loss berechnen: Fehler zwischen Vorhersage 'output' und echten Labels 'batch_y'
        loss = ???

        # 3. Gradienten aus dem letzten Schritt zuruecksetzen
        ???

        # 4. Backward Pass: Gradienten fuer alle Gewichte berechnen
        ???

        # 5. Gewichte anhand der Gradienten anpassen
        ???

        epoch_losses.append(loss.item())

    loss_pro_epoche.append(np.mean(epoch_losses))
    print(f"Epoche {epoch + 1}/{anzahl_epochen}: Loss = {loss_pro_epoche[-1]:.4f}")

print("Training abgeschlossen.")

> 💡 **Good to know:**
> Bei MNIST reichen schon **5 Epochen** für eine sehr gute Test-Accuracy (~97 % laut
> Vorlesungsfolie) — ein deutlicher Unterschied zu den 60 Epochen, die wir in Kapitel 6 für den
> viel kleineren, aber verrauschteren Versicherer-Datensatz gebraucht haben. Bild-Muster wie
> Ziffernformen sind für ein CNN vergleichsweise einfach zu lernen.

### Zwischenfazit

Das Modell ist trainiert. Zeit, ehrlich zu prüfen, wie gut es wirklich ist.

## Schritt 6 — Evaluation: Loss-Kurve, Accuracy, Fehlklassifikationen

Wie in Kapitel 6 schauen wir zuerst auf die **Loss-Kurve**, dann auf die **Test-Accuracy** auf
nie gesehenen Daten.

In [ ]:
# I DO: Loss-Kurve ueber alle Epochen plotten

plt.figure(figsize=(7, 5))
plt.plot(range(1, anzahl_epochen + 1), loss_pro_epoche, color="tab:purple", marker="o")
plt.xlabel("Epoche")
plt.ylabel("Durchschnittlicher Loss (CrossEntropyLoss)")
plt.title("Trainingsverlauf: Loss pro Epoche - SimpleCNN auf MNIST")
plt.show()

Jetzt die Vorhersagen auf dem kompletten Testset — genau wie auf der Vorlesungsfolie "MNIST
trainieren — Evaluation": `torch.max(output.data, 1)` liefert pro Bild die Klasse mit der
höchsten Roh-Ausgabe (Logit).

In [ ]:
# I DO: Test-Accuracy auf dem kompletten Testset berechnen

model.eval()
korrekt = 0
gesamt = 0

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        output = model(batch_x)
        _, vorhersage = torch.max(output.data, 1)   # Klasse mit dem hoechsten Logit pro Bild
        korrekt += (vorhersage == batch_y).sum().item()
        gesamt += batch_y.size(0)

genauigkeit = 100 * korrekt / gesamt
print(f"Test-Accuracy: {genauigkeit:.2f}% ({korrekt} von {gesamt} Testbildern korrekt)")

> 💡 **Good to know:**
> Anders als in Kapitel 6 (starke Klassenimbalance, ~6 % Betrug) sind die 10 Ziffern-Klassen bei
> MNIST **ungefähr gleich häufig** vertreten — hier ist eine hohe Accuracy tatsächlich ein
> aussagekräftiger Indikator für ein gut trainiertes Modell, ganz ohne den Vorbehalt aus Kapitel
> 6.

Eine Confusion Matrix über alle 10 Klassen zeigt, welche Ziffern das Netz am ehesten verwechselt.

In [ ]:
# I DO: Confusion Matrix ueber alle 10 Ziffern-Klassen

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

alle_vorhersagen = []
alle_labels = []

model.eval()
with torch.no_grad():
    for batch_x, batch_y in test_loader:
        output = model(batch_x)
        _, vorhersage = torch.max(output.data, 1)
        alle_vorhersagen.extend(vorhersage.tolist())
        alle_labels.extend(batch_y.tolist())

matrix = confusion_matrix(alle_labels, alle_vorhersagen)
anzeige = ConfusionMatrixDisplay(confusion_matrix=matrix, display_labels=list(range(10)))

fig, achse = plt.subplots(figsize=(8, 8))
anzeige.plot(cmap="Blues", ax=achse, colorbar=False)
plt.title("Confusion Matrix - SimpleCNN auf MNIST-Testdaten")
plt.show()

Zuletzt schauen wir uns ein paar **Fehlklassifikationen** konkret an — das ist oft
aufschlussreicher als jede reine Zahl.

In [ ]:
# I DO: Ein paar Fehlklassifikationen visualisieren

fehler_indizes = [i for i, (v, l) in enumerate(zip(alle_vorhersagen, alle_labels)) if v != l]
print(f"Anzahl Fehlklassifikationen: {len(fehler_indizes)} von {len(alle_labels)}")

fig, achsen = plt.subplots(2, 5, figsize=(10, 4))
for achse, idx in zip(achsen.flat, fehler_indizes[:10]):
    bild, echtes_label = test_dataset[idx]
    achse.imshow(bild.squeeze(), cmap="gray")
    achse.set_title(f"Echt: {echtes_label}, Netz: {alle_vorhersagen[idx]}")
    achse.axis("off")

plt.tight_layout()
plt.show()

> ⚠️ **Common Pitfall:**
> Bei genauem Hinsehen sind viele Fehlklassifikationen bei MNIST auch für einen **Menschen**
> nicht eindeutig — eine schlampig geschriebene 4 kann leicht wie eine 9 aussehen. Ein Modell,
> das ausschließlich bei solchen mehrdeutigen Fällen daneben liegt, ist in der Praxis oft besser
> als die reine Accuracy-Zahl vermuten lässt.

### Zwischenfazit

Wir haben jetzt ein vollständig trainiertes und evaluiertes CNN — von der Architektur über das
Training bis zu einzelnen Fehlklassifikationen. Zurück zu unserem eigentlichen Case: Wie würden
wir das für echte Kfz-Schadensfotos angehen, ohne 60.000 gelabelte Trainingsbilder zu haben?

## Mini-Exercise — Transfer Learning am Kfz-Schadensfoto-Beispiel (konzeptuelles Mock)

**Das Problem:** Ein CNN wie unser `SimpleCNN` von Grund auf für Kfz-Schadensfotos zu trainieren,
würde laut Vorlesungsfolie "Von Grund auf vs. Transfer Learning" **Wochen** dauern und
Zehntausende gelabelter Fotos brauchen — die haben wir hier nicht. Die Praxis-Lösung: **Transfer
Learning**. Wir nehmen ein Netz, das bereits auf Millionen Fotos (ImageNet) trainiert wurde,
frieren die bereits gelernten Conv-Layer ein und trainieren nur einen neuen, kleinen "Kopf"
(Custom Head) auf unsere drei Schadensklassen.

> ⚠️ **Common Pitfall — kein echter Datensatz:**
> Uns liegen keine echten Kfz-Schadensfotos vor. Die Zelle unten arbeitet deshalb bewusst mit
> **zufälligen (synthetischen) Bild-Tensoren** — nur um die PyTorch-**API** von Transfer Learning
> Schritt für Schritt zu zeigen, nicht um ein echtes, sinnvolles Modell zu trainieren! Die
> Vorhersagen, die dabei herauskommen, sind bedeutungslos (Zufallsrauschen rein, Zufallsrauschen
> raus) — es geht rein um den **Code-Mechanismus**.

In [ ]:
# I DO: Vortrainiertes ResNet50 laden und Conv-Layer einfrieren

import torchvision.models as models

# 1. ResNet50 mit ImageNet-Gewichten laden (weights=... ist die moderne Variante von
#    pretrained=True aus der Vorlesungsfolie - torchvision empfiehlt seit Version 0.13 diese Form)
schaden_modell = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

# 2. Alle bestehenden Parameter (die "gelernten Augen" des Netzes) einfrieren:
#    requires_grad = False bedeutet "beim Training NICHT veraendern"
for parameter in schaden_modell.parameters():
    parameter.requires_grad = False

anzahl_eingefroren_gesamt = sum(p.numel() for p in schaden_modell.parameters())
print(f"Parameter im vortrainierten ResNet50 (alle vorerst eingefroren): {anzahl_eingefroren_gesamt:,}")

Jetzt ersetzen wir die letzte Schicht (`model.fc`) — die ursprünglich 1.000 ImageNet-Klassen
vorhersagt — durch einen **Custom Head** für unsere **drei** Schadensklassen: "Kratzer",
"Beule", "Totalschaden".

In [ ]:
# I DO: Custom Head fuer unsere 3 Schadensklassen ersetzen

schaden_klassen = ["Kratzer", "Beule", "Totalschaden"]

# model.fc war urspruenglich: Linear(2048, 1000) - fuer 1000 ImageNet-Klassen.
# Wir ersetzen sie durch einen eigenen kleinen Kopf mit 3 Ausgaben.
schaden_modell.fc = nn.Sequential(
    nn.Linear(2048, 512),
    nn.ReLU(),
    nn.Linear(512, len(schaden_klassen)),
)

# Nur die neuen fc-Parameter sollen trainierbar sein (der Rest bleibt eingefroren)
for parameter in schaden_modell.fc.parameters():
    parameter.requires_grad = True

trainierbare_parameter = sum(p.numel() for p in schaden_modell.parameters() if p.requires_grad)
eingefrorene_parameter = sum(p.numel() for p in schaden_modell.parameters() if not p.requires_grad)
gesamt_parameter = trainierbare_parameter + eingefrorene_parameter

print(f"Trainierbare Parameter (nur der neue Custom Head): {trainierbare_parameter:,}")
print(f"Eingefrorene Parameter (ResNet50-Backbone, bleibt unveraendert): {eingefrorene_parameter:,}")
print(f"-> Nur {100 * trainierbare_parameter / gesamt_parameter:.1f}% aller Parameter werden trainiert.")

Jetzt simulieren wir einen einzigen Trainingsschritt mit **zufälligen** Bild-Tensoren — Größe
`(Batch, 3, 224, 224)`, exakt das RGB-Format, das ResNet50 erwartet — um die komplette
Trainings-API einmal durchlaufen zu sehen.

> 🎯 **Your Task:**
> Ergänze in der Zelle unten die fehlende Zeile, die den Optimizer erstellt. Orientiere Dich an
> der Vorlesungsfolie "Transfer Learning: Implementierung (2/2)" — wichtig: Der Optimizer soll
> **nur** `schaden_modell.fc.parameters()` bekommen, nicht `schaden_modell.parameters()`!

In [ ]:
# =========================================================================
# 🎯 EXERCISE: Optimizer fuer den Custom Head erstellen
# =========================================================================
# Instruction: Ersetze ??? durch einen Adam-Optimizer, der NUR die Parameter
# von schaden_modell.fc trainiert (lr=0.001).

torch.manual_seed(42)

# Synthetische (zufaellige) "Fotos" und "Labels" - NUR zur API-Demonstration, kein echter Datensatz!
mock_fotos = torch.randn(8, 3, 224, 224)          # 8 zufaellige "Fotos", 224x224 RGB
mock_labels = torch.randint(0, 3, (8,))            # 8 zufaellige Labels aus {0, 1, 2}

criterion = nn.CrossEntropyLoss()
optimizer = ???  # <- HIER: Adam-Optimizer, nur fuer schaden_modell.fc.parameters(), lr=0.001

schaden_modell.train()
output = schaden_modell(mock_fotos)
loss = criterion(output, mock_labels)

optimizer.zero_grad()
loss.backward()
optimizer.step()

print(f"Output-Form: {tuple(output.shape)}  (Batch=8, 3 Schadensklassen)")
print(f"Loss nach einem einzigen Trainingsschritt: {loss.item():.4f}")
print("\n(Erinnerung: Da die Eingaben zufaellig sind, ist dieser Loss-Wert selbst bedeutungslos -")
print(" entscheidend ist nur, dass die komplette Trainings-Pipeline fehlerfrei durchlaeuft.)")

> 💡 **Good to know:**
> Der einzige strukturelle Unterschied zum Trainingsloop weiter oben: Der Optimizer bekommt hier
> **nur** `schaden_modell.fc.parameters()` statt aller Modell-Parameter. Weil die restlichen
> ResNet50-Gewichte `requires_grad=False` haben, würde `loss.backward()` für sie ohnehin keine
> Gradienten berechnen — der eingeschränkte Optimizer ist trotzdem die sauberere, explizitere
> Variante.

## Self-Check — Konzeptuelle Reflexion

> 🎯 **Your Task:**
> Beantworte die folgenden zwei Fragen als Kommentare bzw. `print()`-Ausgaben in der Zelle
> unten.
>
> **Frage 1:** Warum lohnt sich für **Bilddaten** (wie unsere Kfz-Schadensfotos) ein CNN
> gegenüber einem einfachen Dense-Netz (wie `VersichererMLP` aus Kapitel 6) so viel mehr als bei
> **Tabellendaten**? Nenne **einen** konkreten Grund.
>
> **Frage 2:** Der Versicherer hat nur **200 gelabelte** Kfz-Schadensfotos zur Verfügung — nicht
> annähernd genug, um ein CNN von Grund auf zu trainieren. Welchen der beiden Ansätze aus diesem
> Kapitel (CNN from scratch vs. Transfer Learning) würdest Du empfehlen, und warum?

In [ ]:
# TODO: Trage Deine zwei Antworten als Kommentare bzw. print()-Ausgaben ein.

antwort_1 = "???"
antwort_2 = "???"

print(f"Antwort 1: {antwort_1}")
print(f"Antwort 2: {antwort_2}")

## Summary & Key Takeaways

- **Convolution** wendet einen kleinen, trainierbaren Filter (Kernel) über das ganze Bild an —
  element-weise Multiplikation + Summe pro Position erzeugt eine **Feature Map**. Genau das haben
  wir am Sobel-Filter-Beispiel `Output = 4` von Hand nachgerechnet.
- **Pooling** (meist Max-Pooling) verdichtet Feature Maps ohne trainierbare Parameter — unser
  4×4-Beispiel wurde per Max-Pool(2×2) zu `[[5,7],[8,6]]`.
- **Parameterfreigabe** ist der Grund, warum CNNs für Bilder so viel effizienter sind als
  Dense-Netze: derselbe 3×3-Filter wird überall wiederverwendet (288 statt 19,3 Mio. Gewichte bei
  einem 224×224-RGB-Bild).
- **`SimpleCNN`**-Architektur: Conv(1→32) → Pool → Conv(32→64) → Pool → Flatten → Dense(128) →
  Dense(10) — räumlicher Verlauf 28×28 → 14×14 → 7×7.
- Der **Trainingsloop** folgt demselben Muster wie in Kapitel 6 (Forward Pass → Loss →
  `zero_grad()` → `backward()` → `step()`), nur mit `CrossEntropyLoss` für **Multiklassen**-Output
  statt `BCEWithLogitsLoss` für binäre Klassifikation.
- **Transfer Learning:** Ein vortrainiertes Netz (z. B. ResNet50 auf ImageNet) einfrieren, nur
  einen neuen Custom Head auf die eigenen Klassen trainieren — spart Wochen an Trainingszeit und
  Zehntausende gelabelte Bilder.
- **Ehrliches Fazit:** MNIST von Grund auf zu trainieren funktioniert hier, weil ein riesiger,
  gelabelter Datensatz (60.000 Bilder) verfügbar ist. Für unseren Kfz-Schadensfoto-Case mit nur
  einer Handvoll gelabelter Fotos ist **Transfer Learning** fast immer der praktikablere Weg.

**Weiter geht's:** Im nächsten Kapitel verlassen wir Bilder — dort geht es um **Natural Language
Processing** und Transformer-Modelle für Freitext aus Schadensmeldungen und Kundenbewertungen.